In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/content/cleaned_data.csv")

In [3]:
df

,id,topic,sentiment,text,tokens,clean_text
0,2401,Borderlands,Positive,i am coming to the borders and i will kill you...,"['coming', 'borders', 'kill']",coming borders kill
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you all,"['im', 'getting', 'borderlands', 'kill']",im getting borderlands kill
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,"['im', 'coming', 'borderlands', 'murder']",im coming borderlands murder
3,2401,Borderlands,Positive,im getting on borderlands and i will murder y...,"['im', 'getting', 'borderlands', 'murder']",im getting borderlands murder
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...,"['im', 'getting', 'borderlands', 'murder']",im getting borderlands murder
...,...,...,...,...,...,...
3206,1757,CallOfDutyBlackopsColdWar,Neutral,i like black ops cold case at its core way bet...,"['like', 'black', 'ops', 'cold', 'case', 'core...",like black ops cold case core way better pile ...
3207,1757,CallOfDutyBlackopsColdWar,Neutral,because i like black ops but cold war at it s...,"['like', 'black', 'ops', 'cold', 'war', 'core'...",like black ops cold war core way better pile s...
3208,1757,CallOfDutyBlackopsColdWar,Neutral,i like black jack and war at its core way bett...,"['like', 'black', 'jack', 'war', 'core', 'way'...",like black jack war core way better pile shit ...
3209,1758,CallOfDutyBlackopsColdWar,Positive,new video my first black ops cold war beta ...,"['new', 'video', 'first', 'black', 'ops', 'col...",new video first black ops cold war beta gamepl...


TF_IDF

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

df['clean_text'] = df['clean_text'].fillna('')

tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting to 5000 features for demonstration
tfidf_matrix = tfidf_vectorizer.fit_transform(df['clean_text'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (3211, 4372)


word2text

In [5]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 48.6 MB/s eta 0:00:00


In [6]:
from gensim.models import Word2Vec

In [7]:
#tokenization
tokenized_text = df['clean_text'].apply(lambda x: x.split())

In [8]:
# Train Word2Vec model

word2vec_model = Word2Vec(sentences=tokenized_text, vector_size=100, window=5, min_count=1, workers=4)

print("Word2Vec model trained successfully.")
print(f"Number of words in vocabulary: {len(word2vec_model.wv)}")
print(f"Vector size: {word2vec_model.wv.vector_size}")

Word2Vec model trained successfully.
Number of words in vocabulary: 4390
Vector size: 100


train using logistic regression

In [9]:
from sklearn.linear_model import LogisticRegression

X = tfidf_matrix
y = df['sentiment']

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

LogisticRegression(max_iter=1000)

cross validation

In [10]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean cross-validation score: {cv_scores.mean():.4f}")
print(f"Standard deviation of cross-validation scores: {cv_scores.std():.4f}")

Cross-validation scores: [0.8911353  0.91121495 0.90186916 0.90498442 0.8894081 ]
Mean cross-validation score: 0.8997
Standard deviation of cross-validation scores: 0.0083


evaluvation

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

  Irrelevant       0.99      0.79      0.88        85
    Negative       0.99      0.85      0.91       131
     Neutral       0.93      0.78      0.85       161
    Positive       0.80      0.99      0.89       266

    accuracy                           0.88       643
   macro avg       0.93      0.85      0.88       643
weighted avg       0.90      0.88      0.88       643

